# Extracting regulatory flags from the US Toxic Substances Control Act (TSCA) Inventory
This notebook demonstrates how to extract identifiers from the TSCA Inventory while preserving the regulatory flags associated with individual substances. The resulting dataset can be used as an input to global inventory compilation while retaining detailed TSCA-specific regulatory context.

**Prerequisites**
- Download the non-CBI CSV archive of the TSCA Inventory from the [US EPA website](https://www.epa.gov/tsca-inventory/how-access-tsca-inventory).

## Step 1: Patch the list-tag parser
The FLAG column contains TSCA-specific regulatory codes. This patch prefixes each flag with *US-TSCA_* so that the values remain clearly tied to the parent legislation.

In [1]:
from gic.fields import list_tag


def patch_list_tag(value: str) -> str:
    """Prefix TSCA regulatory flags while preserving the parent list tag."""
    return value if value == "US-TSCA" else f"US-TSCA_{value}"


list_tag.ListTagField.patch = staticmethod(patch_list_tag)

## Step 2: Import and configure the extractor
Set the input and output paths and map the relevant CSV columns to GIC fields.

In [2]:
from gic import GenericCsvExtractor

# Initialise the extractor.
extractor = GenericCsvExtractor()

# Input and output files.
extractor.input_file = "input/TSCAINV_072025.csv"
extractor.output_file = "output/tsca.jsonl"
extractor.errors_file = "output/tsca_errors.tsv"

# Column-to-field mapping: Chemical name and regulatory flags.
extractor.columns = {
    6: "chemical_name",
    9: "list_tag",
}

# Assign the parent list tag to every extracted row.
extractor.constants = {"list_tag": ["US-TSCA"]}

# File layout settings.
extractor.start_row = 2
extractor.subdelimiter = ";"

## Step 3: Run the extraction and inspect the output
Execute the extractor, write the records to JSONL, and inspect an example row containing multiple regulatory flags.

In [3]:
df = extractor.extract()

# Inspect an example row showing multiple regulatory flags.
print(df[9].as_dict())

{'chemical_name': ['Benzoic acid, 2,6-dichloro-'], 'list_tag': ['US-TSCA_PMN', 'US-TSCA_S', 'US-TSCA_5E', 'US-TSCA']}
